# Visualización y informes

Todo lo que un run produce vive en su directorio (`.soma/runs/<run_id>/`);
todo lo de este notebook son *lectores* de esos ficheros. Tres capas:
lectores/agregación (Rust, `RunView`), overlays sobre el grafo
(mermaid/graphviz anotados) y figuras Plotly (`soma.viz`, extra
`pip install 'somatize[viz]'`).

In [ ]:
import json, pathlib

import soma
from soma import Filter, Graph, Study, search


class Scaler(Filter):
    _cache_version = "nb07-scaler-v1"

    def fit(self, x, y=None):
        return {"mean": sum(x) / len(x)}

    def forward(self, x, state):
        return [v - state["mean"] for v in x]


class Model(Filter):
    _cache_version = "nb07-model-v1"

    def fit(self, x, y=None):
        return {"w": 0.5}

    def forward(self, x, state):
        return [v * state["w"] for v in x]


g = Graph()
g.node("scaler", Scaler())
g.node("model", Model())
g.connect("scaler", "model")

with g.track_run("nb07-fit", kind="fit", tags=["demo"]) as run:
    g.fit([1.0, 2.0, 3.0, 4.0])
    for step in range(6):
        run.log("loss", 1.0 / (step + 1), step=step)
        run.log("val_f1", 0.6 + 0.06 * step, step=step)

run.id

## Listar e inspeccionar runs

`soma.runs()` escanea `<root>/runs/` (más reciente primero). Un run
`running` con heartbeat caducado se reporta como `crashed`. Cada
`RunView` agrega el log de eventos en formas listas para graficar.

In [ ]:
for r in soma.runs():
    print(f"{r.id:32s} {r.kind:6s} {r.state:10s} {r.name}")

view = soma.runs()[0]
view.node_timings()

In [ ]:
view.cache_activity(), view.metric_series("val_f1")[:2]

## Arquitectura anotada

El mermaid del grafo con el overlay del run: duración por nodo, hits de
caché y flags de salud, con clases de estado (`soma_completed`,
`soma_cached`, `soma_failed`, `soma_flagged`). Pégalo en mermaid.live o
míralo renderizado en `soma report`.

In [ ]:
print(view.to_mermaid())

## Figuras

Métodos instalados sobre `RunView`; devuelven figuras Plotly
interactivas (requieren el extra `somatize[viz]`).

In [ ]:
view.plot_metrics()

In [ ]:
view.plot_gantt()  # dónde se fue el tiempo de pared, nodo a nodo

## Optimización estilo Optuna

Un `Study` es también un run (su run-dir tiene `study.json`); las
figuras llevan los nombres de Optuna para no sorprender a nadie.

In [ ]:
def objective(trial):
    for step in range(5):
        f1 = 0.5 + 0.35 * trial["lr"] ** 0.2 * (step + 1) / 5
        if trial.report("f1", f1, step):
            return None
    return None


study = Study(
    "nb07-hpo",
    search_space=[
        {"type": "float", "name": "lr", "low": 1e-4, "high": 1e-1, "scale": "log"},
        {"type": "float", "name": "dropout", "low": 0.0, "high": 0.5},
    ],
    strategy="bayesian",
    n_trials=8,
    objectives=[("f1", "maximize")],
    pruning=("median", 2),
    seed=42,
)
study.run(objective)
study.best_trial["id"], study.best_trial["metrics"]

In [ ]:
study.plot_optimization_history()

In [ ]:
study.plot_parallel_coordinate()

In [ ]:
study.plot_timeline()

In [ ]:
study.trials_dataframe().head()

## Informe HTML

Un fichero autocontenido por run: DAG anotado, tiles de eficiencia,
curvas, sección HPO con tabla de trials y sección de salud. Desde la
CLI: `soma report <run_id>` (`--inline` embebe plotly.js para verlo sin
red). Los blobs `<script id="soma-data-*">` embebidos son el contrato
que leerá la futura GUI.

In [ ]:
html = study.to_html(path="nb07_report.html")
print(f"nb07_report.html: {len(html) / 1024:.0f} KiB")
print("secciones:", [s.split("</h2>")[0] for s in html.split("<h2>")[1:]])